# TNBC miRNA–mRNA GAT study, phase 3 (notebook 2 of 2): outer-fold evaluation of the selected edge-forcing variant

**Post-hoc, added after the frozen study.** Run this only after notebook 1's `selected_variant.json` has been committed
as `kaggle/phase3_frozen/selected_variant.json` on `kaggle-phase3r`. The notebook checks that the file is committed and
unmodified and stops otherwise, or if no variant passed the planted control (then there is nothing to evaluate).

| Arm | What |
|---|---|
| Selected variant, miRDB edges | 16 configurations (seed 0) on the inner-validation split, then seeds 1–9 of the best, 5 repeats × 5 folds |
| Same variant, rewired edges and no cross edges | configuration of the miRDB arm, 5 seeds |

Contrasts (Holm over 3 × 2 modalities): variant_mirdb − joint_nocross (frozen phase-1 arm), − variant_rewired,
− variant_nocross; ΔF1 with Nadeau–Bengio CIs, TOST ±0.03, power, sign-flip p.

**Settings:** Internet on, accelerator None (CPU). Smoke test first, then *Save & Run All*. If it stops at the time
budget, attach this notebook's output and run again (`restore` reuses every finished fit).

In [ ]:
import os, subprocess, time

SMOKE_TEST = True          # True: one fold, one seed per arm type and a runtime estimate. False: everything.
WORKERS = os.cpu_count()   # parallel fits; each fit is single-threaded (frozen setting)
TIME_BUDGET_H = 11.0       # Kaggle sessions stop at 12 h
0.5_H = 0.5        # kept for baselines, analysis and the zip; no new GAT fit starts after budget - reserve

BRANCH = 'kaggle-phase3r'
REPO_URL = 'https://github.com/heramn/TNBC-miRNA-mRNA-GNN.git'
SRC = '/tmp/TNBC-miRNA-mRNA-GNN'
PY = '/tmp/p3env/bin/python'
WORK = '/kaggle/working'
T0 = time.time()
DEADLINE = T0 + (TIME_BUDGET_H - 0.5_H) * 3600
os.environ.update(STUDY_RUNS=f'{WORK}/phase3_runs', PHASE3_OUT=f'{WORK}/phase3_output', STUDY_CACHE='/tmp/study_cache',
                  OMP_NUM_THREADS='1', OPENBLAS_NUM_THREADS='1', MKL_NUM_THREADS='1')


def sh(cmd, check=True):
    print('$', cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    if p.wait() and check:
        raise RuntimeError(f'exit {p.returncode}: {cmd}')
    return p.returncode

SM = '--smoke' if SMOKE_TEST else ''
print('workers:', WORKERS, '| smoke test:', SMOKE_TEST)

## 1. Clone the repository at `kaggle-phase3r`

In [ ]:
sh(f'rm -rf {SRC} && git clone -q -b {BRANCH} {REPO_URL} {SRC} && git -C {SRC} log -1 --format="%h %s"')

## 2. Python environment

The same isolated Python 3.12 venv as phase 2, with phase 1's pins: torch **2.15.0+cpu**, torch_geometric **2.8.1**,
scikit-learn **1.7.2** (1.8 changes the grouped splits). Kaggle's preinstalled CUDA torch is left alone.

In [ ]:
sh('pip install -q uv')
sh('uv venv -q --python 3.12 /tmp/p3env')
sh(f'uv pip install -q --python {PY} numpy==2.5.5 pandas==3.1.1 scipy==1.18.1 scikit-learn==1.7.2 statsmodels==0.15.0 '
   'matplotlib==3.11.2 xgboost==3.4.1 pydeseq2==0.6.0 anndata==0.13.4 torch_geometric==2.8.1 pytest')
sh(f'uv pip install -q --python {PY} torch==2.15.0 --index-url https://download.pytorch.org/whl/cpu')
sh(f'{PY} -c "import sys, torch, torch_geometric, sklearn, numpy; print(sys.version.split()[0], torch.__version__, '
   'torch_geometric.__version__, sklearn.__version__, numpy.__version__)"')

## 3. Data, restore, frozen-file tests, frozen selection check

In [ ]:
os.chdir(SRC)
sh(f'{PY} kaggle/phase3.py data')
sh(f'{PY} kaggle/phase3.py restore')
sh(f'{PY} -m pytest -q -p no:cacheprovider study/tests')
sh(f'{PY} kaggle/phase3.py check-frozen')

## 4. Outer-fold fits (stage 1: 16 configurations at seed 0; stage 2: the rest with the selected configuration)

In [ ]:
sh(f'{PY} kaggle/phase3.py run --part eval {SM} --workers {WORKERS} --deadline {DEADLINE:.0f}')

## 5. Analysis and RESULTS rows (full mode)

In [ ]:
if SMOKE_TEST:
    print('smoke test: analysis skipped')
else:
    rc = sh(f'{PY} kaggle/phase3.py analyze --part eval', check=False)
    print('analysis', 'done' if rc == 0 else f'not complete (exit {rc}); attach this output and rerun to finish the missing fits')
    sh(f'{PY} kaggle/phase3.py report', check=False)

## 6. Zip the outputs

In [ ]:
import shutil, zipfile
os.chdir(WORK)
z = shutil.make_archive(f'{WORK}/tnbc_phase3b', 'zip', WORK, '.')
with zipfile.ZipFile(z) as zf:
    names = [n for n in zf.namelist() if n.startswith('phase3_runs/') and n.endswith('.csv')]
print(z, f'{os.path.getsize(z) / 1e6:.1f} MB,', len(names), 'per-run CSVs')
rows = f'{WORK}/phase3_output/RESULTS_phase3_rows.md'
if os.path.exists(rows):
    print(open(rows).read())
print(f'total time {(time.time() - T0) / 3600:.2f} h')